# Favorita daily sales forecasting

Start with the prediction task, input keys and temporal split. This notebook documents the selected seven-day-token, three-year-history Transformer. Recorded test WAPE is 49.9963%.

In [ ]:
from IPython.display import display
from pathlib import Path
import json, sys, subprocess
ROOT = Path.cwd()
if ROOT.name == "favorita":
    ROOT = ROOT.parent
assert (ROOT / "favorita" / "train.py").exists()
sys.path.insert(0, str(ROOT / "favorita"))

## Prepare the data

Obtain the competition train.csv using the README link. Install CUDA-enabled PyTorch and favorita/requirements.txt. The preparation step aggregates store, product and date, preserving chronological order.

In [ ]:
TRAIN_CSV = ROOT / "data/favorita/train.csv"
PANEL = ROOT / "work/favorita-panel.parquet"
# Run once after downloading:
# subprocess.run([sys.executable, str(ROOT / "favorita/prepare.py"),
#                 "--train", str(TRAIN_CSV)], cwd=ROOT, check=True)

## Inspect input schema

Negative aggregate sales are clipped to zero during loading. Missing rows after series launch become zero. Earlier dates are masked.

In [ ]:
import pyarrow.parquet as pq
if PANEL.exists():
    panel = pq.ParquetFile(PANEL)
    print(panel.schema_arrow)
    print("Rows:", panel.metadata.num_rows)
    display(next(panel.iter_batches(batch_size=5)).to_pandas())
else:
    print("Prepare the panel to inspect local data.")

## Define the experiment

Fit targets end June 20, 2017. Validation is June 21–July 18. Test is July 19–August 15. Each forecast uses actual sales through the previous day, including previous actual holdout observations. This is rolling one-day-ahead evaluation. Fit target eligibility begins at max(28, first observation + 1).

In [ ]:
import pandas as pd
display(pd.DataFrame([["Fit", "2013-01-01", "2017-06-20"],
                      ["Validation", "2017-06-21", "2017-07-18"],
                      ["Test", "2017-07-19", "2017-08-15"]],
                     columns=["Split", "Panel/window start", "End"]))

## Construct the model

157 tokens represent 1,099 historical days. Each seven-day token preserves its daily values through concatenation and projection. Width 128, 12 layers, 8 heads and global historical attention.

In [ ]:
import torch
from retail_patch import PatchTransformer
torch.manual_seed(42)
model = PatchTransformer(7,174685,True,7,sales_history=True,patch_count=157,layers=12)
parameters = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {parameters:,}")
assert parameters == 7240641

## Run full training

All parameters train for two epochs. AdamW learning rate 0.001, batch 1,024, BF16 autocast, L1 loss. Validation WAPE selects weights. Full training can take multiple days. Set the flag deliberately.

In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    subprocess.run([sys.executable, str(ROOT / "favorita/train.py"),
                    "favorita", "long"], cwd=ROOT, check=True)

## Inspect recorded results

WAPE sums absolute errors over all scored item-store-day rows and divides by total actual sales. These metrics and plots come from the completed run. Running these cells reads the bundled artifacts.

In [ ]:
metrics = json.loads((ROOT / "favorita/metrics.json").read_text())
print("Validation WAPE", format(metrics["validation_wape"], ".4%"))
print("Test WAPE", format(metrics["test_wape"], ".4%"))
display(pd.DataFrame([metrics["surge"],metrics["drop"]],index=["Upward","Downward"]))

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / "favorita/forecast.png")))

## Improve the recorded 49.9963% WAPE

The recorded test score is our reference. Improvement experiments use validation data only. We first reproduce the fixed checkpoint on validation, compare simple sales-history baselines, and examine signed errors and high-volume series.

We then fit a scale correction and a blend with the trailing seven-day mean on the first 14 validation days. The next 14 validation days assess transfer across time. The test dates remain reserved for a single final assessment after decisions are frozen.

These cells test whether the current forecast is systematically too high or low and whether recent demand complements the long-context model. They establish a measured starting point before another full training run.


In [ ]:
import numpy as np
import time

RUN_VALIDATION_ANALYSIS = False
ANALYSIS_BATCH_SIZE = 32
ANALYSIS_DIR = ROOT / "runs" / "favorita-improvement"
# These optional cells load approximately 1.2 GB of dense host arrays and
# perform millions of validation predictions. Enable on a GPU with spare capacity.
# Output files are separate from the recorded checkpoint and metrics.


## Load the original panel and verify the checkpoint mapping

The mapping stored with the checkpoint is essential because each embedding row identifies a specific item-store pair. We apply the same treatment of negative sales, missing daily rows and pre-launch history as training.

Panel checks run only when the analysis flag is enabled. Predictions use the checkpoint in evaluation mode and the original FP32 evaluation path, with a smaller evaluation batch to limit memory use.


In [ ]:
if RUN_VALIDATION_ANALYSIS:
    assert torch.cuda.is_available(), "Validation inference requires an available CUDA GPU."
    assert PANEL.exists(), "Prepare the Favorita panel first."
    ids = np.load(ROOT / "favorita" / "series.npy", allow_pickle=False)
    assert ids.shape == (174685, 2)
    index = pd.MultiIndex.from_arrays(ids.T)
    first_day = np.datetime64("2013-01-01", "D").astype(int)
    end_day = np.datetime64("2017-08-15", "D").astype(int)
    span = int(end_day - first_day + 1)
    raw = np.zeros((len(ids), span), dtype=np.float32)
    first = np.full(len(ids), span, dtype=np.int32)
    promotion = np.zeros((len(ids), 28), dtype=bool)
    fit_end = span - 56
    test_start = span - 28
    panel = pq.ParquetFile(PANEL)
    assert panel.metadata.num_rows == metrics["raw_panel_rows"]
    for batch in panel.iter_batches(columns=["store", "product", "period", "sales", "promotion"]):
        frame = batch.to_pandas()
        row = index.get_indexer(pd.MultiIndex.from_frame(frame[["store", "product"]]))
        assert (row >= 0).all(), "Panel identity differs from the published checkpoint."
        day = frame["period"].to_numpy().astype("datetime64[D]").astype(int) - first_day
        raw[row, day] = np.maximum(frame["sales"].to_numpy(), 0)
        np.minimum.at(first, row, day)
        selected = (day >= fit_end) & (day < test_start)
        promotion[row[selected], day[selected] - fit_end] = (
            frame.loc[selected, "promotion"].fillna(0).to_numpy() != 0
        )
    assert np.unique(ids, axis=0).shape == ids.shape
    seen = first < fit_end
    scale = float(
        np.sum(raw[:, :fit_end], dtype=np.float64)
        / np.maximum(fit_end - first, 0).sum()
    )
    del frame, panel
    print("Panel loaded. Scale:", scale, "Eligible series:", int(seen.sum()))


## Reproduce validation inference and history baselines

Predict each validation day using actual history up to the preceding day. A seven-day seasonal naive baseline repeats the same weekday from the previous week. A trailing-mean baseline averages the previous seven days.

All three predictions share the same eligible rows. Baseline use of earlier actual validation observations matches the checkpoint's rolling protocol. No future target values enter any input.


In [ ]:
if RUN_VALIDATION_ANALYSIS:
    from retail_patch import make_dates
    device = torch.device("cuda")
    model = model.to(device)
    model.load_state_dict(torch.load(
        ROOT / "favorita" / "model.pt", map_location=device, weights_only=True
    ))
    model.eval()
    calendar = make_dates(span, device)
    offsets = np.arange(-1099, 0)
    prediction = np.empty((len(ids), 28), np.float32)
    started = time.perf_counter()
    with torch.no_grad():
        for column, day in enumerate(range(fit_end, test_start)):
            for start in range(0, len(ids), ANALYSIS_BATCH_SIZE):
                series = np.arange(start, min(start + ANALYSIS_BATCH_SIZE, len(ids)))
                past = day + offsets
                values = raw[series[:, None], np.maximum(past, 0)].copy()
                missing_days = past[None, :] < first[series, None]
                values[missing_days] = np.nan
                missing_tokens = missing_days.reshape(-1, 157, 7).all(axis=-1)
                missing_tokens[:, -1] = False
                output = model(
                    calendar[torch.as_tensor(np.maximum(past, 0), device=device)]
                        .unsqueeze(0).expand(len(series), -1, -1),
                    calendar[day].expand(len(series), -1),
                    torch.as_tensor(series, device=device),
                    torch.as_tensor(values / scale, device=device),
                    torch.as_tensor(missing_tokens, device=device),
                )
                prediction[series, column] = output.cpu().numpy() * scale
            print(f"Validation day {column + 1}/28, elapsed {(time.perf_counter()-started)/60:.1f} min")
    assert np.isfinite(prediction).all()
    actual = raw[:, fit_end:test_start]
    seasonal = raw[:, fit_end - 7:test_start - 7]
    trailing_mean = np.column_stack([
        raw[:, day - 7:day].mean(axis=1)
        for day in range(fit_end, test_start)
    ])
    mask = np.broadcast_to(seen[:, None], actual.shape)
    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        ANALYSIS_DIR / "validation-predictions.npz",
        ids=ids, actual=actual, prediction=prediction, seasonal=seasonal,
        trailing_mean=trailing_mean, seen=seen, promotion=promotion
    )
    model = model.cpu()
    torch.cuda.empty_cache()


## Measure bias and identify where errors concentrate

Positive signed bias means overprediction. WAPE by segment divides segment absolute error by segment actual sales. Its contribution column uses total validation actual sales so segment contributions add to overall WAPE.

Volume groups use fit-period sales only. Promotion groups describe observed validation conditions and help assess whether promotion inputs could be valuable. Their causal effect cannot be established from this grouping.


In [ ]:
CACHE = ANALYSIS_DIR / "validation-predictions.npz"
if CACHE.exists():
    with np.load(CACHE, allow_pickle=False) as saved:
        ids = saved["ids"]
        actual = saved["actual"]
        prediction = saved["prediction"]
        seasonal = saved["seasonal"]
        trailing_mean = saved["trailing_mean"]
        seen = saved["seen"]
        promotion = saved["promotion"]
    mask = np.broadcast_to(seen[:, None], actual.shape)

    def scores(y, forecast, selected):
        denominator = y[selected].sum(dtype=np.float64)
        if denominator <= 0:
            return {"wape": np.nan, "signed_bias": np.nan, "mae": np.nan}
        error = forecast[selected] - y[selected]
        return {
            "wape": np.abs(error).sum(dtype=np.float64) / denominator,
            "signed_bias": error.sum(dtype=np.float64) / denominator,
            "mae": np.abs(error).mean(dtype=np.float64),
        }

    comparison = pd.DataFrame({
        "Transformer": scores(actual, prediction, mask),
        "Seasonal naive lag 7": scores(actual, seasonal, mask),
        "Trailing mean 7": scores(actual, trailing_mean, mask),
    }).T
    display(comparison)
    print("Recorded validation WAPE:", metrics["validation_wape"])
    print("Reproduction difference:",
          comparison.loc["Transformer", "wape"] - metrics["validation_wape"])
    comparison.to_csv(ANALYSIS_DIR / "validation-baselines.csv")

    total_sales = actual[mask].sum(dtype=np.float64)
    segments = []
    for label, selected in [("Promotion", mask & promotion),
                            ("No promotion", mask & ~promotion)]:
        record = scores(actual, prediction, selected)
        record.update(
            segment=label, rows=int(selected.sum()),
            contribution=np.abs(actual[selected]-prediction[selected]).sum(dtype=np.float64)/total_sales
        )
        segments.append(record)
    display(pd.DataFrame(segments).set_index("segment"))
    if "raw" in globals():
        fit_sales = raw[:, :fit_end].sum(axis=1, dtype=np.float64)
        boundaries = np.quantile(fit_sales[seen], [0.5, 0.9])
        volume_group = np.digitize(fit_sales, boundaries, right=True)
        rows = []
        for group, label in enumerate(["Bottom 50%", "Next 40%", "Top 10%"]):
            selected = mask & (volume_group[:, None] == group)
            record = scores(actual, prediction, selected)
            record["fit_volume_group"] = label
            record["contribution"] = np.abs(actual[selected]-prediction[selected]).sum(dtype=np.float64)/total_sales
            rows.append(record)
        display(pd.DataFrame(rows).set_index("fit_volume_group"))
else:
    print("Enable validation analysis to create the prediction cache.")


## Fit two small changes before retraining

Candidate 1 scales predictions by a single multiplier. Candidate 2 blends the Transformer with the trailing seven-day mean, optionally with scale correction. The fixed candidate grid is selected using June 21–July 4 only.

July 5–18 is the confirmation period. Report both periods independently. Prefer the simplest candidate that improves confirmation WAPE without worsening the event metrics relevant to the application. This short confirmation split has sampling uncertainty; another chronological backtest is needed before claiming general improvement.


In [ ]:
if CACHE.exists():
    calibration = mask.copy()
    calibration[:, 14:] = False
    confirmation = mask.copy()
    confirmation[:, :14] = False
    candidates = []
    for weight in [0.0, 0.25, 0.5, 0.75, 1.0]:
        for multiplier in [0.8, 0.9, 1.0, 1.1, 1.2]:
            combined = multiplier * (
                weight * prediction + (1.0 - weight) * trailing_mean
            )
            candidates.append({
                "transformer_weight": weight,
                "multiplier": multiplier,
                "calibration_wape": scores(actual, combined, calibration)["wape"],
            })
    candidate_table = pd.DataFrame(candidates)
    selected = candidate_table.sort_values("calibration_wape", kind="stable").iloc[0]
    combined = selected["multiplier"] * (
        selected["transformer_weight"] * prediction
        + (1.0-selected["transformer_weight"]) * trailing_mean
    )
    report = pd.DataFrame([
        {"method": "Original", "calibration_wape": scores(actual,prediction,calibration)["wape"],
         "confirmation_wape": scores(actual,prediction,confirmation)["wape"]},
        {"method": "Selected scale/blend",
         "calibration_wape": scores(actual,combined,calibration)["wape"],
         "confirmation_wape": scores(actual,combined,confirmation)["wape"]},
    ]).set_index("method")
    display(selected.to_frame("Selected on first 14 days"))
    display(report)
    candidate_table.to_csv(ANALYSIS_DIR / "calibration-grid.csv", index=False)
    report.to_csv(ANALYSIS_DIR / "confirmation-report.csv")
    experiment = {
        "transformer_weight": float(selected["transformer_weight"]),
        "multiplier": float(selected["multiplier"]),
        "selection_period": ["2017-06-21","2017-07-04"],
        "confirmation_period": ["2017-07-05","2017-07-18"],
        "test_assessed": False,
    }
    (ANALYSIS_DIR / "candidate.json").write_text(json.dumps(experiment,indent=2))


## Inspect daily and store-level errors

A low total error can coexist with large errors for particular stores or days. Daily plots show the sum of sales across eligible series. Store WAPE uses all scored products and days in each store. Use the table to choose diagnostic examples before inspecting test data.


In [ ]:
if CACHE.exists():
    import matplotlib.pyplot as plt
    dates = pd.date_range("2017-06-21", periods=28)
    figure, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
    axes[0].plot(dates, actual[seen].sum(axis=0), label="Actual")
    axes[0].plot(dates, prediction[seen].sum(axis=0), label="Transformer")
    axes[0].plot(dates, trailing_mean[seen].sum(axis=0), label="Trailing mean 7")
    axes[0].legend()
    axes[0].set_ylabel("Total daily sales")
    daily_wape = [
        scores(actual[:, i],prediction[:, i],seen)["wape"] for i in range(28)
    ]
    axes[1].plot(dates, daily_wape)
    axes[1].axvline(dates[14], color="gray", linestyle="--", label="Confirmation begins")
    axes[1].set_ylabel("Daily WAPE")
    axes[1].legend()
    figure.savefig(ANALYSIS_DIR / "validation-diagnostics.png", dpi=150)
    display(figure)
    plt.close(figure)
    rows = []
    for store in np.unique(ids[:,0]):
        selected = mask & (ids[:,0,None] == store)
        record = scores(actual,prediction,selected)
        record["store"] = int(store)
        rows.append(record)
    by_store = pd.DataFrame(rows).set_index("store").sort_values("wape",ascending=False)
    display(by_store)
    by_store.to_csv(ANALYSIS_DIR / "validation-by-store.csv")


## Next full-training experiments

Keep the 157-token context, 12 layers, eight heads, fitting dates and validation protocol fixed when testing each input change. Use the unchanged model as the control.

| Order | Single change | Reason to test | Acceptance evidence |
|---|---|---|---|
| 1 | Scale correction / trailing-mean blend above | Check level bias and short-term adaptation without retraining | Improvement on the untouched 14-day confirmation period |
| 2 | Historical promotion plus forecast-day promotion where known in advance | Predict scheduled changes in demand | Lower validation WAPE and lower promotion-segment error |
| 3 | Per-series scale from fitting data, with a global fallback | Accommodate different sales levels across products and stores | Improvement across volume segments without harming high-volume WAPE |
| 4 | Lag-1, lag-7, lag-14, lag-28 and trailing-mean residual prediction | Give a direct route for strong recent-demand signals | Improvement over both the control Transformer and simple history baselines |
| 5 | Training-only holiday, store and product metadata | Add known local and product structure | Improvement on chronological validation with feature ablations |

Promotion availability must match what would actually be known at each prediction origin. Never supply future actual sales. The current 28-day validation has already been used for checkpoint selection, so confirmation results are diagnostic. Stronger evidence requires another chronological backtest with all preprocessing fitted within its own training cutoff.

After selecting a candidate, freeze its configuration, retrain if required, and evaluate the original July 19–August 15 test period once. Record WAPE, signed bias, upward/downward TP/FP/FN and runtime. Updating only the token length and store sharding simultaneously would confound the explanation of any improvement.

No improved WAPE is claimed by this notebook until these optional analysis experiments have been executed.
